# Istio Ambient, end to end — a petshop walkthrough

**Solo Enterprise for Istio, ambient mode, on kind.** One app (a petshop), every ambient security feature, and the Solo Enterprise differences called out as we go.

What we build, in the order the mesh actually enforces it:

| Step | Layer | What it shows |
|------|-------|---------------|
| Identity = certificate | — | every workload gets a SPIFFE SVID from its ServiceAccount |
| L4 authorization | **L4 · ztunnel** | allow/deny on the peer's identity, no waypoint |
| Access logs | **L4 · ztunnel** | identity-aware allow/deny logs (`src.identity`) |
| The shared-SA gap | **L4** | two pods on one ServiceAccount are indistinguishable |
| A waypoint | **L7 · waypoint** | opt-in Envoy for HTTP concerns |
| An IdP + JWT | — | Keycloak issues tokens (`alice` user, `bob` admin) |
| JWT authorization | **L7 · waypoint** | `RequestAuthentication` + claim-based `AuthorizationPolicy` |
| Solo Enterprise extras | — | L7 telemetry from ztunnel (no waypoint), 1.30 workload-claims |

> **Where JWTs live:** a request JWT is HTTP, so it is validated and authorized at the **waypoint (L7)**. ztunnel at **L4** authorizes on the workload's **certificate identity**, never the user's token. This notebook shows both, and where each belongs.

Every step below shows the real command and the real manifest — nothing hidden behind a wrapper script.


## Prerequisites

- `docker`, `kind`, `kubectl`, `helm`, `istioctl`, `jq`, and an **authenticated `gcloud`** (Solo Istio images pull from `us-docker.pkg.dev`).
- `SOLO_ISTIO_LICENSE_KEY` exported (or `SECRETS_FILE` pointing at a file that exports it).

**Edition:** Enterprise — the mesh is the Solo distribution, installed by the Gloo Operator. Validated on Solo Istio `1.29.3-solo`, Gloo Operator `0.5.2`, Gateway API `v1.5.1`.


In [ ]:
# Run from the lab directory. Constants used throughout.
[ -d istio-ambient-cert-identity-kind ] && cd istio-ambient-cert-identity-kind || :
export CTX=kind-cert-identity
export ISTIO_NS=istio-system
# license (edit if your secrets live elsewhere)
export SECRETS_FILE="${SECRETS_FILE:-$HOME/code/solo/secrets/secrets-envs.sh}"
[ -f "$SECRETS_FILE" ] && set -a && . "$SECRETS_FILE" && set +a
echo "context: $CTX ; license set: $([ -n "$SOLO_ISTIO_LICENSE_KEY" ] && echo yes || echo NO)"


## 1 · Stand up the mesh

Ambient is one control-plane switch plus two node components. We show every command; `scripts/setup-cluster.sh` runs exactly these if you prefer one line.


In [ ]:
# 1a. kind cluster (1 control-plane + 1 worker)
kind create cluster --config kind/cluster.yaml


In [ ]:
# 1b. Gateway API CRDs (and drop the safe-upgrades policy so the operator can manage its own)
kubectl --context $CTX apply --server-side -f \
  https://github.com/kubernetes-sigs/gateway-api/releases/download/v1.5.1/standard-install.yaml
kubectl --context $CTX delete validatingadmissionpolicybinding safe-upgrades.gateway.networking.k8s.io --ignore-not-found
kubectl --context $CTX delete validatingadmissionpolicy        safe-upgrades.gateway.networking.k8s.io --ignore-not-found


In [ ]:
# 1c. Pre-pull the Solo Istio images on the host (gcloud-authenticated) and load them into kind.
#     (docker save | kind load, because kind's containerd chokes on multi-arch indexes.)
REG=us-docker.pkg.dev/soloio-img/istio ; VER=1.29.3
case "$(uname -m)" in arm64|aarch64) PLAT=linux/arm64;; *) PLAT=linux/amd64;; esac
for img in pilot proxyv2 install-cni ztunnel; do
  docker pull -q $REG/$img:$VER
  docker save --platform $PLAT $REG/$img:$VER -o /tmp/$img.tar
  kind load image-archive /tmp/$img.tar --name cert-identity && rm -f /tmp/$img.tar
done


In [ ]:
# 1d. Gloo Operator, and the Solo Istio license as a Secret
helm --kube-context $CTX upgrade --install gloo-operator \
  oci://us-docker.pkg.dev/solo-public/gloo-operator-helm/gloo-operator \
  -n gloo-system --create-namespace --version 0.5.2 --wait

kubectl --context $CTX create namespace $ISTIO_NS --dry-run=client -o yaml | kubectl --context $CTX apply -f -
kubectl --context $CTX -n $ISTIO_NS create secret generic solo-istio-license \
  --from-literal=license="$SOLO_ISTIO_LICENSE_KEY" --dry-run=client -o yaml | kubectl --context $CTX apply -f -


The whole control + data plane is one CR. `.spec.cluster` becomes the mesh **trust domain**, so identities are `spiffe://cert-identity/ns/<ns>/sa/<sa>` — **not** `cluster.local`.


In [ ]:
# 1e. ServiceMeshController — istiod + istio-cni + ztunnel, in ambient mode
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: operator.gloo.solo.io/v1
kind: ServiceMeshController
metadata: { name: managed-istio, namespace: gloo-system }
spec:
  version: "1.29.3"
  dataplaneMode: Ambient
  distribution: Standard
  scalingProfile: Demo
  cluster: cert-identity
EOF

# wait for istiod, then wire the license env and turn on JSON access logs
kubectl --context $CTX -n $ISTIO_NS rollout status deploy/istiod --timeout=300s
kubectl --context $CTX -n $ISTIO_NS patch deployment istiod --type=json -p='[{"op":"add",
  "path":"/spec/template/spec/containers/0/env/-","value":{"name":"SOLO_LICENSE_KEY",
  "valueFrom":{"secretKeyRef":{"name":"solo-istio-license","key":"license"}}}}]' || true
kubectl --context $CTX -n $ISTIO_NS patch daemonset ztunnel --type=json -p='[{"op":"add",
  "path":"/spec/template/spec/containers/0/env/-","value":{"name":"LOG_FORMAT","value":"json"}}]' || true
kubectl --context $CTX -n $ISTIO_NS rollout status daemonset/ztunnel --timeout=180s
kubectl --context $CTX -n $ISTIO_NS rollout status daemonset/istio-cni-node --timeout=180s
echo "ambient ready, trust domain cert-identity, JSON access logs on"


## 2 · The petshop app

One namespace, enrolled in ambient with a single label. The cast:

| Workload | ServiceAccount → identity | Role |
|---|---|---|
| `petstore` | `sa/petstore` | the API (`GET /pets`, `DELETE /pets/{id}`) |
| `storefront` | `sa/storefront` | client the L4 policy **allows** |
| `analytics` | `sa/analytics` | client the L4 policy **denies** |
| `checkout-blue` | `sa/checkout` | shares one SA with green … |
| `checkout-green` | `sa/checkout` | … so both present the **same** SVID |


In [ ]:
# Enrol the namespace in ambient — one label, no restarts.
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: v1
kind: Namespace
metadata:
  name: petshop
  labels:
    istio.io/dataplane-mode: ambient
EOF

# Deploy the petshop (manifests in yaml/10-app/ — petstore is the API, the rest are clients)
kubectl --context $CTX apply -f yaml/10-app/
kubectl --context $CTX -n petshop rollout status deploy/petstore deploy/storefront deploy/analytics deploy/checkout-blue deploy/checkout-green --timeout=180s
kubectl --context $CTX -n petshop get pods


## 3 · Identity is the certificate

ztunnel holds one mTLS SVID per workload identity. The URI SAN is the ServiceAccount — and that is exactly what an L4 policy matches on. Watch the gap: `checkout-blue` and `checkout-green` share `sa/checkout`, so ztunnel holds **one** cert for both.


In [ ]:
# Which ztunnel is on the node with our pods, then the SVIDs it holds
ZT=$(kubectl --context $CTX -n $ISTIO_NS get pod -l app=ztunnel \
      --field-selector spec.nodeName=cert-identity-worker -o jsonpath='{.items[0].metadata.name}')
istioctl --context $CTX ztunnel-config certificate "$ZT.$ISTIO_NS" | grep -E "CERTIFICATE NAME|petshop" | grep -Ei "name|leaf"


**Captured live** — one leaf SVID per ServiceAccount, and a single `sa/checkout` cert for the two checkout pods:

```
CERTIFICATE NAME                                  TYPE   STATUS      VALID CERT   ...
spiffe://cert-identity/ns/petshop/sa/analytics    Leaf   Available   true
spiffe://cert-identity/ns/petshop/sa/checkout     Leaf   Available   true      <- shared by blue AND green
spiffe://cert-identity/ns/petshop/sa/petstore     Leaf   Available   true
spiffe://cert-identity/ns/petshop/sa/storefront   Leaf   Available   true
```


## 4 · Authorize on identity, at L4

One `AuthorizationPolicy`, enforced by ztunnel, no waypoint. It selects `petstore` and allows only the `storefront` identity — so it simultaneously **allows storefront** and **denies analytics and checkout** (ztunnel fails closed: once any ALLOW selects a workload, everything unnamed is denied).

Principals use the trust domain `cert-identity`, **not** `cluster.local`.


In [ ]:
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: allow-storefront, namespace: petshop }
spec:
  selector: { matchLabels: { app: petstore } }
  action: ALLOW
  rules:
  - from: [{ source: { principals: ["cert-identity/ns/petshop/sa/storefront"] } }]
    to:   [{ operation: { ports: ["8080"] } }]
EOF
sleep 10
# each client curls petstore every 2s and prints the HTTP code
for d in storefront analytics checkout-blue checkout-green; do
  echo "$d: $(kubectl --context $CTX -n petshop logs deploy/$d --tail=1)"
done


**Captured live** — the identity decides, with no change to any app:

```
storefront:     storefront -> GET petstore/pets : 200
analytics:      analytics -> GET petstore/pets : 000000     <- denied
checkout-blue:  checkout-blue -> GET petstore/pets : 000000  <- denied
checkout-green: checkout-green -> GET petstore/pets : 000000 <- denied
```


## 5 · Read the L4 access logs

ztunnel logs every connection with the peer SPIFFE identities and the outcome — identity-aware audit at L4, no waypoint.


In [ ]:
ZT=$(kubectl --context $CTX -n $ISTIO_NS get pod -l app=ztunnel \
      --field-selector spec.nodeName=cert-identity-worker -o jsonpath='{.items[0].metadata.name}')
# a DENY (analytics) and an ALLOW (storefront), projected to the fields that matter
kubectl --context $CTX -n $ISTIO_NS logs "$ZT" --tail=500 | grep petshop \
 | grep -E 'policy rejection|http_access' | tail -4 \
 | jq -rc '{src:(.["src.identity"]//"-"), dst:(.["dst.service"]//"-"), method:(.method//"L4"), code:(.response_code//"-"), error:(.error//"ok")}'


**Captured live** — the denied line names the caller's identity and the reason; the allowed line even carries HTTP fields (more on that in §10):

```
{"src":"spiffe://cert-identity/ns/petshop/sa/analytics","dst":"petstore.petshop...","method":"L4","code":"-","error":"connection closed due to policy rejection: allow policies exist, but none allowed"}
{"src":"spiffe://cert-identity/ns/petshop/sa/storefront","dst":"petstore.petshop...","method":"GET","code":200,"error":"ok"}
```


## 6 · The shared-ServiceAccount ceiling

Add `sa/checkout` to the allowed set. Both `checkout-blue` and `checkout-green` get in — and there is **no L4 rule that lets one in and keeps the other out**, because they present the same SVID. This is the limit of SA-scoped identity, and the motivation for §11.


In [ ]:
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: allow-checkout, namespace: petshop }
spec:
  selector: { matchLabels: { app: petstore } }
  action: ALLOW
  rules:
  - from: [{ source: { principals: ["cert-identity/ns/petshop/sa/checkout"] } }]
    to:   [{ operation: { ports: ["8080"] } }]
EOF
sleep 10
for d in storefront analytics checkout-blue checkout-green; do
  echo "$d: $(kubectl --context $CTX -n petshop logs deploy/$d --tail=1)"
done
# -> storefront 200, checkout-blue 200, checkout-green 200, analytics still 000000


## 7 · Add a waypoint for L7

Everything so far was L4. HTTP concerns — methods, paths, JWTs — need a waypoint: a standalone Envoy you opt into per service. We reset the L4 policies here so the L7 story stands on its own (in production you would keep both, defence in depth).


In [ ]:
# reset L4 policies for a clean L7 demo
kubectl --context $CTX -n petshop delete authorizationpolicy allow-storefront allow-checkout --ignore-not-found

# a waypoint (istio-waypoint GatewayClass), then enrol the petstore Service to use it
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: gateway.networking.k8s.io/v1
kind: Gateway
metadata:
  name: petstore-waypoint
  namespace: petshop
  labels: { istio.io/waypoint-for: service }
spec:
  gatewayClassName: istio-waypoint
  listeners:
  - name: mesh
    port: 15008
    protocol: HBONE
EOF
kubectl --context $CTX label service petstore -n petshop istio.io/use-waypoint=petstore-waypoint --overwrite
kubectl --context $CTX -n petshop rollout status deploy/petstore-waypoint --timeout=150s


## 8 · An IdP, and a real JWT

Keycloak runs in its own (non-ambient) namespace with a `petshop` realm and two users: **alice** (`role: user`) and **bob** (`role: admin`). We mint a token with the password grant and read its claims — the issuer and `realm_access.roles` are what the waypoint will check.


In [ ]:
kubectl --context $CTX apply -f yaml/40-idp/keycloak.yaml
kubectl --context $CTX -n keycloak rollout status deploy/keycloak --timeout=240s


In [ ]:
# mint + decode a token (run from a petshop pod; it reaches Keycloak cross-namespace)
KC=http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop/protocol/openid-connect/token
tok() { kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "curl -s -m10 -d grant_type=password -d client_id=petshop -d username=$1 -d password=$1 -d scope=openid $KC" \
  | sed -n 's/.*"access_token":"\([^"]*\)".*/\1/p'; }
BOB=$(tok bob)
echo "$BOB" | cut -d. -f2 | { read p; python3 - "$p" <<'PY'
import sys,base64,json
p=sys.argv[1]; p+="="*(-len(p)%4)
d=json.loads(base64.urlsafe_b64decode(p))
print("iss:  ", d["iss"]); print("user: ", d["preferred_username"]); print("roles:", d["realm_access"]["roles"])
PY
}


**Captured live:**

```
iss:   http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop
user:  bob
roles: ['admin', 'user']
```


## 9 · Authorize on the JWT, at L7

Two objects on the waypoint. `RequestAuthentication` validates the token against Keycloak's JWKS. `AuthorizationPolicy` then requires a valid token for **any** request and — with a CEL `when` clause over the nested `realm_access.roles` claim — restricts `DELETE` to admins.


In [ ]:
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: RequestAuthentication
metadata: { name: petshop-jwt, namespace: petshop }
spec:
  targetRefs:
  - { kind: Gateway, group: gateway.networking.k8s.io, name: petstore-waypoint }
  jwtRules:
  - issuer:  "http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop"
    jwksUri: "http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop/protocol/openid-connect/certs"
    forwardOriginalToken: true
---
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: petstore-jwt-authz, namespace: petshop }
spec:
  targetRefs:
  - { kind: Gateway, group: gateway.networking.k8s.io, name: petstore-waypoint }
  action: ALLOW
  rules:
  - from: [{ source: { requestPrincipals: ["*"] } }]           # any valid token may read
    to:   [{ operation: { methods: ["GET"] } }]
  - from: [{ source: { requestPrincipals: ["*"] } }]           # only admins may delete
    to:   [{ operation: { methods: ["DELETE"] } }]
    when: [{ key: "request.auth.claims[realm_access][roles]", values: ["admin"] }]
EOF
sleep 10


In [ ]:
# the matrix: no token, alice (user), bob (admin)
KC=http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop/protocol/openid-connect/token
tok() { kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "curl -s -m10 -d grant_type=password -d client_id=petshop -d username=$1 -d password=$1 -d scope=openid $KC" \
  | sed -n 's/.*"access_token":"\([^"]*\)".*/\1/p'; }
call() { kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c "$1" ; }
ALICE=$(tok alice); BOB=$(tok bob)
echo "no token    GET  /pets     -> $(call "curl -s -o /dev/null -w '%{http_code}' -m5 http://petstore:8080/pets")"
echo "alice       GET  /pets     -> $(call "curl -s -o /dev/null -w '%{http_code}' -m5 -H 'Authorization: Bearer $ALICE' http://petstore:8080/pets")"
echo "alice(user) DELETE /pets/1 -> $(call "curl -s -o /dev/null -w '%{http_code}' -m5 -X DELETE -H 'Authorization: Bearer $ALICE' http://petstore:8080/pets/1")"
echo "bob(admin)  DELETE /pets/1 -> $(call "curl -s -o /dev/null -w '%{http_code}' -m5 -X DELETE -H 'Authorization: Bearer $BOB' http://petstore:8080/pets/1")"


**Captured live** — the claim decides the write:

```
no token    GET  /pets     -> 403
alice       GET  /pets     -> 200
alice(user) DELETE /pets/1 -> 403
bob(admin)  DELETE /pets/1 -> 200
```


## 10 · What Solo Enterprise adds

**L7 telemetry from ztunnel, with no waypoint.** On the Solo distribution, ztunnel emits HTTP metrics, access logs and traces itself — which is why the *allowed* L4 line back in §5 already carried `method`/`path`/`response_code`. Upstream OSS ztunnel logs are L4-only.


In [ ]:
ZT=$(kubectl --context $CTX -n $ISTIO_NS get pod -l app=ztunnel \
      --field-selector spec.nodeName=cert-identity-worker -o jsonpath='{.items[0].metadata.name}')
istioctl --context $CTX ztunnel-config all "$ZT.$ISTIO_NS" -o json | jq '.config.l7Config'


**Captured live:**

```json
{ "enabled": true,
  "access_log": { "enabled": true, "skip_connection_log": false },
  "metrics":    { "enabled": true },
  "tracing":    { "enabled": true, "otlp_endpoint": "http://opentelemetry-collector.istio-system:4317" } }
```

Other Solo Enterprise ambient differentiators worth knowing:
- **Multicluster ambient** — east-west gateway (`istio-eastwest`), `istioctl multicluster expose`, global-service segments — so identity and JWT policy span clusters.
- **agentgateway as the L7 proxy** — a `Gateway` with `gatewayClassName: enterprise-agentgateway` brings MCP/LLM/AI policy to mesh traffic.
- **Lifecycle & packaging** — Gloo Operator (`ServiceMeshController`), FIPS builds, air-gap, Solo UI.
- **Per-workload cert claims + CEL at L4** — the 1.30 preview in §11.


## 11 · Closing the gap — 1.30 workload claims (reference)

§6 hit a wall: two pods on one ServiceAccount are indistinguishable at L4. The Solo **1.30 line** (alpha, Enterprise) closes it — CEL over claims baked into the *certificate*, still at L4, still no waypoint. This lab runs 1.29.3-solo, so this is **reference only** (`yaml/30-reference/workload-claims-1.30.yaml`), not applied here.

```yaml
# 1) turn it on:  ENABLE_WORKLOAD_CLAIMS=true on ztunnel  (per-pod certs)
# 2) annotate the pod — the claim is embedded in its mTLS cert at issuance:
metadata:
  annotations:
    solo.io.security-claims/tier: "gold"     # e.g. checkout-blue=gold, checkout-green=silver
# 3) authorize at L4 on the claim, with CEL:
spec:
  action: ALLOW
  rules:
  - when:
    - key: "source.claims['solo.io.security-claims.tier']"   # '/' -> '.'
      values: ["gold"]
```

The SPIFFE URI never changes; the claim rides alongside it, so ztunnel can now tell `checkout-blue` from `checkout-green` — the exact thing §6 could not do. Note the distinction from §9: **this** CEL is over the workload *certificate*; the JWT CEL in §9 is over the user's *request token* at the waypoint.


## 12 · L4 or L7? the decision

| You want to authorize on … | Layer | Where | Waypoint? |
|---|---|---|---|
| Which **workload** (its identity) is calling | L4 | ztunnel | no |
| Source namespace / IP / destination port | L4 | ztunnel | no |
| Per-workload **cert claims** (1.30) | L4 | ztunnel | no |
| A **user's JWT** and its claims | L7 | waypoint | **yes** |
| HTTP method / path / header | L7 | waypoint | **yes** |

L4 is always on and free; reach for a waypoint the moment the decision is about HTTP or an end-user token.


In [ ]:
# tidy up
kind delete cluster --name cert-identity
